- This script takes the negatives from the test set of IMMREP and splits them to create a test and validation dataset.
- Furthermore, it adds these to the existing test and val datasets and additionally adds a 'binding_flag' column to classify a positive or negative result.

To do in this script:
1. Split negatives into val and test, create two csv files
2. Concatenate current val and test to inckude new negatives - extending the pair_id column and adding a column for 'binding_flag'
3. Update val_negatives.csv and test_negatives.csv to extend the old pair_id column
4. Take binding_flag = 0 for val and tes, create yaml files for val_negatives and test_negatives for Boltz runs (but using the new pair_id), creating MSA files
5. Make the YAML files sit under current data/test and data/val folders, following the chunk logic for Boltz runs
5. Create manifests for val_negative and test_negative for Boltz runs, again using the new pair_id
6. (Maybe) extend val and test manifests to include new negatives so have just three manifests in total, depending on how data classes use these/how I change the data structure for the new autoencoder runs

N.B.:
- There are 2 unique HLAs, 20 unique pepties, around 1,000 unique TCRs
- 5,000 rows have one HLA and 5,000 rows have the other
- Need to take some random yaml files from train/val/test to make sure they match to the pair_id I have created in the folders


#### 0. Imports and Paths

In [1]:
from pathlib import Path
import subprocess, shutil, os
import pandas as pd  # you use pd in cell 3
import os, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import shutil

In [2]:
# === Paths ===
#CSV_PATH = "data/raw/HLA/full_positives_hla_seq.csv"  # update if needed
CSV_PATH = "/home/natasha/multimodal_model/data/raw/immrep2025_test_set.csv"
VAL_PATH = "/home/natasha/multimodal_model/data/val/val_df_clean.csv"
TEST_PATH = "/home/natasha/multimodal_model/data/test/test_df_clean.csv"

BASE_DIR  = Path("/home/natasha/multimodal_model") #/ "data" / "raw"
MSA_DIR   = BASE_DIR / "data" / "raw" / "MSA"
TRAIN_DIR = BASE_DIR / "data" / "train"
VAL_DIR   = BASE_DIR / "data" / "val"
TEST_DIR  = BASE_DIR / "data" / "test"
MANI_DIR  = BASE_DIR / "data" / "manifests"

# add current csv files of test and val for later merging
# n.b. be extremely careful to not overwrite the current csv files

# create directories if they don't exist
MSA_DIR.mkdir(parents=True, exist_ok=True)
MANI_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

# === Load CSV and preview ===
df = pd.read_csv(CSV_PATH)
#df.head(3)


In [3]:
# -------- Paths --------
#BASE_DIR = Path("/home/natasha/multimodal_model")
DB_COMBINED = BASE_DIR / "data" / "raw" / "MSA" / "big_combo_subset_tcrs_50000_w_mhc_seqs.fasta"

# use the split DBs created
DBS = {
    "tcra": BASE_DIR / "data" / "raw" / "MSA" / "db_split" / "alpha.fasta",
    "tcrb": BASE_DIR / "data" / "raw" / "MSA" / "db_split" / "beta.fasta",
    "mhc":  BASE_DIR / "data" / "raw" / "MSA" / "db_split" / "mhc.fasta",
}
# def pick_db_for(stem: str) -> Path:
#     return DBS.get(stem, DB_COMBINED)


# OUT_ROOT = BASE_DIR / "data" / "raw" / "MSA" / "jackhmmer_msas"
# OUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------- Controls --------
VERBOSE = False              # set False to silence prints
KEEP_INTERMEDIATES = False   # set False to delete .sto/.tbl/.a3m after filtering

# -------- jackhmmer / filtering parameters --------
JACK_ITERS = 1
EVALUE = 1e-10
CPU_THREADS = 4
CPU_THREADS = os.cpu_count() or 4

MAX_SEQS = 64
# keep identity de-dup very relaxed so hhfilter mostly just caps:
ID_THR_DEFAULT = 100        # 100 = no ID-based collapse
COV_THR_TCR     = 50
COV_THR_MHC     = 30

def have(cmd): return shutil.which(cmd) is not None
if VERBOSE:
    print("Deps:",
          "jackhmmer" if have("jackhmmer") else "MISSING",
          "esl-reformat" if have("esl-reformat") else ("reformat.pl" if have("reformat.pl") else "MISSING"),
          "hhfilter" if have("hhfilter") else "MISSING")


#### 1. Split Data into Validate and Test

In [4]:
# IMPORTANT - NEED TO TAKE LABEL = 0 FOR NEGATIVES

In [5]:
# ============================================================
# 1. Pre-processing
# ============================================================

def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    """Clean up TCR chains, build TCR_full and masks."""
    df = df.copy()

    # 1. Fill empty/nan values with 'X' token
    df['TCRa'] = df['TCRa'].fillna('X')
    df['TCRb'] = df['TCRb'].fillna('X')

    # Replace empty strings with 'X'
    df.loc[df['TCRa'] == '', 'TCRa'] = 'X'
    df.loc[df['TCRb'] == '', 'TCRb'] = 'X'

    # Build TCR_full and alpha/beta masks
    df['TCR_full'] = df['TCRa'] + df['TCRb']
    df['m_alpha'] = 1
    df['m_beta'] = 1
    df.loc[df['TCRa'] == 'X', 'm_alpha'] = 0
    df.loc[df['TCRb'] == 'X', 'm_beta'] = 0

    # Remove rows with invalid TCR_full entries
    df = df.dropna(subset=['TCR_full'])
    df = df[df['TCR_full'] != ' ']
    df = df[df['TCR_full'] != 'nan']

    return df


In [7]:
# columns needed: 'hla_sequence', 'peptide', 'tcra_trimmed', 'tcrb_trimmed'

# === Load CSV and preview ===
df = pd.read_csv(CSV_PATH)

# take label == 0 for negatives
df = df[df['label'] == 0]

df = df.reset_index(drop=True)
df["orig_row"] = df.index.astype(int)
df["pair_id_negative"] = df["orig_row"].map(lambda i: f"negative_pair_{i:06d}")
# pair id needs to be carried on from other datasets
# so need to read val and test manifest and add pair_id to df? 
# make new pair_id then when merge add pair_id from other datasets

print(df['label'].unique())

df['TCRa'] = df['tcra_trimmed']
df['TCRb'] = df['tcrb_trimmed']
df = df.rename(columns={"peptide": "Peptide", "hla_sequence": "HLA_sequence"})

df = preprocess_df(df)


df = df[['TCR_full', 'Peptide', 'HLA_sequence', 'TCRa', 'TCRb']]

# clean df 

df.columns
#print(df['HLA'].unique()[0], df['HLA'].unique()[1])

# 9,000 negatives

[0]


Index(['TCR_full', 'Peptide', 'HLA_sequence', 'TCRa', 'TCRb'], dtype='object')

In [29]:
# split into test and val
# test_df has 2190 positives
# val_df has 1960 positives
# pick 2000 random rows using hla 1 for test and 2000 using hla 2 for val
rng = np.random.default_rng(42)

df_test_negatives = df[df["HLA_sequence"] == "MRVTAPRTVLLLLSAALALTETWAGSHSMRYFHTAMSRPGRGEPRFITVGYVDDTLFVRFDSDATSPRKEPRAPWIEQEGPEYWDRETQISKTNTQTYRESLRNLRGYYNQSEAGSHTLQRMYGCDVGPDGRLLRGHNQYAYDGKDYIALNEDLRSWTAADTAAQISQRKLEAARVAEQLRAYLEGECVEWLRRYLENGKDKLERADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDRTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWEPSSQSTVPIVGIVAGLAVLAVVVIGAVVAAVMCRRKSSGGKGGSYSQAACSDSAQGSDVSLTA"]

df_val_negatives = df[df["HLA_sequence"] == "MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWEPSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSSDRKGGSYSQAASSDSAQGSDVSLTACKV"]

print(len(df_test_negatives['Peptide'].unique()))
print(len(df_val_negatives['Peptide'].unique()))

# sample 2000 rows for val and test

# start with val
# one per peptide
base = (
    df_val_negatives
    .groupby("Peptide", group_keys=False)
    .apply(lambda g: g.sample(n=1, random_state=rng))
)

# sample the remaining rows to reach 2000 (if needed)
remaining = df_val_negatives.drop(base.index)
need = max(0, 2000 - len(base))
extra = remaining.sample(n=need, random_state=rng) if need > 0 else remaining.iloc[0:0]

df_val_negatives = pd.concat([base, extra]).sample(frac=1, random_state=rng)


# do the same for test
base = (
    df_test_negatives
    .groupby("Peptide", group_keys=False)
    .apply(lambda g: g.sample(n=1, random_state=rng))
)

# sample the remaining rows to reach 2000 (if needed)
remaining = df_test_negatives.drop(base.index)
need = max(0, 2000 - len(base))
extra = remaining.sample(n=need, random_state=rng) if need > 0 else remaining.iloc[0:0]

df_test_negatives = pd.concat([base, extra]).sample(frac=1, random_state=rng)

# reset index
df_test_negatives = df_test_negatives.reset_index(drop=True)
df_val_negatives = df_val_negatives.reset_index(drop=True)

df_test_negatives["orig_row"] = df_test_negatives.index.astype(int)
df_val_negatives["orig_row"] = df_val_negatives.index.astype(int)

df_test_negatives["pair_id_negative"] = df_test_negatives["orig_row"].map(lambda i: f"negative_pair_{i:06d}")
df_val_negatives["pair_id_negative"] = df_val_negatives["orig_row"].map(lambda i: f"negative_pair_{i:06d}")

df_test_negatives = df_test_negatives[['TCR_full', 'Peptide', 'HLA_sequence', 'pair_id_negative', 'TCRa', 'TCRb']]
df_val_negatives = df_val_negatives[['TCR_full', 'Peptide', 'HLA_sequence', 'pair_id_negative', 'TCRa', 'TCRb']]

df_val_negatives['binding_flag'] = 0
df_test_negatives['binding_flag'] = 0
df_val_negatives['binding_flag'] = 0


print(len(df_test_negatives))
print(len(df_val_negatives))


10
10
2000
2000


/tmp/ipykernel_2238678/3415480286.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=rng))
/tmp/ipykernel_2238678/3415480286.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=rng))


In [ ]:
# save the negatives I am not using for other purposes, at least to keep track of what I have and don't have

In [30]:
overlap = set(df_val_negatives['Peptide']).intersection(set(df_test_negatives['Peptide']))

print(overlap, set(df_val_negatives['Peptide']), set(df_test_negatives['Peptide']))


set() {'SQFNWTIYL', 'HLDDYPYLM', 'ILHTHVPEV', 'FLDCKSYIL', 'KLYPFLWFA', 'TVYPYGTSL', 'MTDYDYLEV', 'YLFNADIWI', 'ILMHATYFL', 'YMFYDGYDV'} {'YENGSTPVL', 'MENWSALEL', 'MEMPDYLLL', 'YESYIPGAL', 'SEESAFYVL', 'FEAQPGALL', 'GEVLACYAL', 'YEDGVIFYL', 'GENALTYAL', 'REDDYSVWL'}


#### 2. Concatenate negatives onto positive val and test dataframes

In [31]:
# because of the way the data is structured, we can take half 

df_val_positives = pd.read_csv(VAL_PATH)
df_test_positives = pd.read_csv(TEST_PATH)

df_val_positives['binding_flag'] = 1
df_test_positives['binding_flag'] = 1

df_val = pd.concat([df_val_positives, df_val_negatives])
df_test = pd.concat([df_test_positives, df_test_negatives])

df_val

,pair_id,Peptide,HLA_sequence,TCR_full,binding_flag,pair_id_negative,TCRa,TCRb
0,pair_000,ASFRPELAEFW,MRVTAPRTVLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...,GEDVEQSLFLSVREGDSSVINCTYTDSSSTYLYWYKQEPGAGLQLL...,1,NaN,NaN,NaN
1,pair_001,ASFSPELRMAW,MRVTAPRTVLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...,GEDVEQSLFLSVREGDSSVINCTYTDSSSTYLYWYKQEPGAGLQLL...,1,NaN,NaN,NaN
2,pair_002,GSLAPEIRMYW,MRVTAPRTVLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...,GEDVEQSLFLSVREGDSSVINCTYTDSSSTYLYWYKQEPGAGLQLL...,1,NaN,NaN,NaN
3,pair_003,GTIRPEIPDYF,MRVTAPRTVLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...,GEDVEQSLFLSVREGDSSVINCTYTDSSSTYLYWYKQEPGAGLQLL...,1,NaN,NaN,NaN
4,pair_004,GTIRPEIREMW,MRVTAPRTVLLLLWGAVALTETWAGSHSMRYFYTAMSRPGRGEPRF...,GEDVEQSLFLSVREGDSSVINCTYTDSSSTYLYWYKQEPGAGLQLL...,1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1995,NaN,TVYPYGTSL,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,QKEVEQNSGPLSVPEGAIASLNCTYSDRGSQSFFWYRQYSGKSPEL...,0,negative_pair_001995,QKEVEQNSGPLSVPEGAIASLNCTYSDRGSQSFFWYRQYSGKSPEL...,DTGVSQNPRHKITKRGQNVTFRCDPISEHNRLYWYRQTLGQGPEFL...
1996,NaN,YLFNADIWI,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,AQSVSQHNHHVILSEAASLELGCNYSYGGTVNLFWYVQYPGQHLQL...,0,negative_pair_001996,AQSVSQHNHHVILSEAASLELGCNYSYGGTVNLFWYVQYPGQHLQL...,DTGVSQNPRHKITKRGQNVTFRCDPISEHNRLYWYRQTLGQGPEFL...
1997,NaN,MTDYDYLEV,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,GNSVTQMEGPVTLSEEAFLTINCTYTATGYPSLFWYVQYPGEGLQL...,0,negative_pair_001997,GNSVTQMEGPVTLSEEAFLTINCTYTATGYPSLFWYVQYPGEGLQL...,DSGVTQTPKHLITATGQRVTLRCSPRSGDLSVYWYQQSLDQGLQFL...
1998,NaN,YLFNADIWI,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,DQQVKQNSPSLSVQEGRISILNCDYTNSMFDYFLWYKKYPAEGPTF...,0,negative_pair_001998,DQQVKQNSPSLSVQEGRISILNCDYTNSMFDYFLWYKKYPAEGPTF...,DTEVTQTPKHLVMGMTNKKSLKCEQHMGHRAMYWYKQKAKKPPELM...


In [32]:
# reset index, fillna with index for pair_id to match structure of previous files
# save as two separate csv files

df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


df_test["index_row"] = df_test.index.astype(int)
df_val["index_row"] = df_val.index.astype(int)

mask = df_test["pair_id"].isna()
df_test.loc[mask, "pair_id"] = df_test.loc[mask, "index_row"].map(
    lambda i: f"negative_pair_{i:06d}"
)

mask = df_val["pair_id"].isna()
df_val.loc[mask, "pair_id"] = df_val.loc[mask, "index_row"].map(
    lambda i: f"negative_pair_{i:06d}"
)

df_test.drop(columns=['index_row'], inplace=True)
df_val.drop(columns=['index_row'], inplace=True)


# check all the pair_ids are unique before saving
print(df_val['pair_id'].nunique(), len(df_val), df_test['pair_id'].nunique(), len(df_test))


df_val.to_csv(BASE_DIR / 'data' / 'val' / 'val_df_clean_pos_neg.csv', index=False)
df_test.to_csv(BASE_DIR / 'data' / 'test' / 'test_df_clean_pos_neg.csv', index=False)

3960 3960 4190 4190


#### 3. Make YAML files with MSAs for Boltz runs

3a) Create new YAML files and manifests for negatives

In [33]:
# take only negatives from val and test

df_val_negatives = df_val[df_val['binding_flag'] == 0]
df_test_negatives = df_test[df_test['binding_flag'] == 0]


print(len(df_val_negatives), len(df_test_negatives))

2000 2000


In [35]:
# --------- Build MSA functions ---------

from __future__ import annotations

import os, re, shlex, shutil, subprocess, textwrap
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, List

import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed

# ----------------------------
# Config
# ----------------------------
@dataclass
class MSAConfig:
    out_root: Path
    db_combined: Path
    dbs: Dict[str, Path]  # {"tcra": ..., "tcrb": ..., "mhc": ...}

    verbose: bool = False
    keep_intermediates: bool = False

    jack_iters: int = 1
    evalue: float = 1e-10
    cpu_threads: int = os.cpu_count() or 4

    max_seqs: int = 64
    id_thr: int = 100
    cov_thr_tcr: int = 50
    cov_thr_mhc: int = 30


def have(cmd: str) -> bool:
    return shutil.which(cmd) is not None


def run(cmd: List[str]):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = p.communicate()
    return p.returncode, out, err


def pick_db_for(cfg: MSAConfig, stem: str) -> Path:
    return cfg.dbs.get(stem, cfg.db_combined)


# ----------------------------
# Sequence cleaning / presence
# ----------------------------
def clean_seq(s) -> str:
    if not isinstance(s, str):
        return ""
    return re.sub(r"[^A-Za-z]", "", s).upper()


def has_seq(s: str, min_len: int = 1) -> bool:
    return isinstance(s, str) and len(s) >= min_len


MISSING_TOKENS = {"", "<unk>", "<UNK>", "UNK", "UNKNOWN", "NA", "N/A", "NONE", "NULL", "NAN"}


def normalise_seq_for_yaml(s) -> str:
    s0 = "" if not isinstance(s, str) else s.strip()
    if s0 in {"<unk>", "<UNK>"}:
        return ""
    s = clean_seq(s0)
    return "" if s in MISSING_TOKENS else s


# ----------------------------
# MSA helpers
# ----------------------------
def sto_to_a3m(sto_path: Path, a3m_path: Path) -> bool:
    if have("esl-reformat"):
        code, out, err = run(["esl-reformat", "a3m", str(sto_path)])
        if code == 0 and out:
            a3m_path.write_text(out)
            return True
    if have("reformat.pl"):
        code, out, err = run(["reformat.pl", "sto", "a3m", str(sto_path), str(a3m_path)])
        return code == 0 and a3m_path.exists()
    return False


def count_a3m(p: Path) -> int:
    if not p.exists():
        return -1
    return sum(1 for ln in p.open() if ln.startswith(">"))


def hhfilter_cap(cfg: MSAConfig, in_a3m: Path, out_a3m: Path, cov_thr: int) -> bool:
    if not have("hhfilter"):
        if cfg.verbose:
            print("WARN: hhfilter not found; copying input → output")
        shutil.copyfile(in_a3m, out_a3m)
        return True

    code, out, err = run(["hhfilter", "-h"])
    use_maxseq = ("-maxseq" in (out or "")) or ("-maxseq" in (err or ""))

    cmd = [
        "hhfilter",
        "-i", str(in_a3m),
        "-o", str(out_a3m),
        "-id", str(cfg.id_thr),
        "-cov", str(cov_thr),
    ]
    cmd += (["-maxseq", str(cfg.max_seqs)] if use_maxseq else ["-n", str(cfg.max_seqs)])

    if cfg.verbose:
        print("[CMD]", " ".join(map(str, cmd)))
    code, out, err = run(cmd)
    if cfg.verbose and err:
        print("[HHFILTER][stderr]\n", err.strip()[:800])
    return code == 0 and out_a3m.exists()


def build_msa_for_chain(cfg: MSAConfig, seq: str, out_dir: Path, stem: str) -> Optional[Path]:
    seq = normalise_seq_for_yaml(seq)
    if not has_seq(seq, min_len=1):
        return None

    out_dir.mkdir(parents=True, exist_ok=True)
    qfa = out_dir / f"{stem}.fa"
    qfa.write_text(f">{stem}\n{seq}\n")

    sto = out_dir / f"{stem}.sto"
    raw_a3m = out_dir / f"{stem}.a3m"
    filt_a3m = out_dir / f"{stem}.filt.a3m"
    tbl = out_dir / f"{stem}.tbl"

    db_fasta = pick_db_for(cfg, stem)

    cmd = [
        "jackhmmer",
        "-N", str(cfg.jack_iters),
        "-A", str(sto),
        "--tblout", str(tbl),
        "-E", str(cfg.evalue),
        "--incE", str(cfg.evalue),
        "--incdomE", str(cfg.evalue),
        "--cpu", str(cfg.cpu_threads),
        str(qfa), str(db_fasta),
    ]
    if cfg.verbose:
        print(f"\n=== {stem} ===")
        print("[CMD]", " ".join(map(str, cmd)))

    code, out, err = run(cmd)

    if code != 0 or (not sto.exists()) or sto.stat().st_size < 200:
        if cfg.verbose:
            print("WARN: bad/empty .sto → fallback to single-seq A3M")
            if err:
                print("[JACK][stderr]\n", err.strip()[:800])
        raw_a3m.write_text(f">{stem}\n{seq}\n")
        raw = raw_a3m
    else:
        ok = sto_to_a3m(sto, raw_a3m)
        if not ok:
            if cfg.verbose:
                print("WARN: sto->a3m failed; using single-seq fallback")
            raw_a3m.write_text(f">{stem}\n{seq}\n")
        raw = raw_a3m

    cov_thr = cfg.cov_thr_tcr if stem in ("tcra", "tcrb") else cfg.cov_thr_mhc
    ok = hhfilter_cap(cfg, raw, filt_a3m, cov_thr=cov_thr)

    if cfg.verbose:
        print("[CHK] filt a3m:", filt_a3m, "nseq=", count_a3m(filt_a3m))

    if not cfg.keep_intermediates:
        for p in (sto, tbl, raw_a3m, qfa):
            try:
                p.unlink()
            except Exception:
                pass

    return filt_a3m if ok else None


def relpath_from_base(p: Path, base: Path) -> str:
    return str(p.relative_to(base))


def make_yaml(
    base_dir: Path,
    tcra_seq: str,
    tcrb_seq: str,
    pep: str,
    mhc_seq: str,
    tcra_msa: Optional[Path],
    tcrb_msa: Optional[Path],
    mhc_msa: Optional[Path],
) -> str:
    def msa_field(p: Optional[Path]) -> str:
        return "empty" if p is None else relpath_from_base(p, base_dir)

    tcra_seq = normalise_seq_for_yaml(tcra_seq)
    tcrb_seq = normalise_seq_for_yaml(tcrb_seq)
    pep      = normalise_seq_for_yaml(pep)
    mhc_seq  = normalise_seq_for_yaml(mhc_seq)

    lines = ["version: 1", "sequences:"]

    def add(pid: str, seq: str, msa: str):
        lines.append(textwrap.dedent(f"""\
        - protein:
            id: {pid}
            sequence: {seq}
            msa: {msa}
        """).rstrip())

    if tcra_seq:
        add("A", tcra_seq, msa_field(tcra_msa))
    if tcrb_seq:
        add("B", tcrb_seq, msa_field(tcrb_msa))
    if pep:
        add("C", pep, "empty")
    if mhc_seq:
        add("D", mhc_seq, msa_field(mhc_msa))

    return "\n".join(lines) + "\n"


def get_pair_id(row, i: int, pad: int = 3) -> str:
    pid = row.get("pair_id", None)
    if isinstance(pid, str) and pid.strip():
        return pid.strip()
    return f"pair_{i:0{pad}d}"


def _yaml_done(yml_path: Path) -> bool:
    return yml_path.exists() and yml_path.stat().st_size > 50


def _msas_done(pair_msa_dir: Path, stems=("tcra", "tcrb", "mhc")) -> bool:
    for st in stems:
        if (pair_msa_dir / f"{st}.filt.a3m").exists():
            continue
        return False
    return True


def _process_one_row(args):
    (i, row_dict, split_name, base_dir_str, yaml_dir_str, msa_root_str, cfg, pad) = args
    base_dir = Path(base_dir_str)
    yaml_dir = Path(yaml_dir_str)
    msa_root = Path(msa_root_str)

    pair_id = get_pair_id(row_dict, i=i, pad=pad)

    yml_path = yaml_dir / f"{pair_id}.yaml"
    pair_msa_dir = msa_root / split_name / pair_id

    pep  = normalise_seq_for_yaml(row_dict.get("Peptide", ""))
    mhc  = normalise_seq_for_yaml(row_dict.get("HLA_sequence", ""))
    tcra = normalise_seq_for_yaml(row_dict.get("TCRa", ""))
    tcrb = normalise_seq_for_yaml(row_dict.get("TCRb", ""))

    tcra_a3m = build_msa_for_chain(cfg, tcra, pair_msa_dir, "tcra")
    tcrb_a3m = build_msa_for_chain(cfg, tcrb, pair_msa_dir, "tcrb")
    mhc_a3m  = build_msa_for_chain(cfg, mhc,  pair_msa_dir, "mhc")

    yml_text = make_yaml(
        base_dir=base_dir,
        tcra_seq=tcra,
        tcrb_seq=tcrb,
        pep=pep,
        mhc_seq=mhc,
        tcra_msa=tcra_a3m,
        tcrb_msa=tcrb_a3m,
        mhc_msa=mhc_a3m,
    )
    yml_path.write_text(yml_text)
    return pair_id, True, ""


def process_split_parallel_resume(
    df: pd.DataFrame,
    split_name: str,
    base_dir: Path,
    yaml_dir: Path,
    msa_root: Path,
    cfg: MSAConfig,
    *,
    pad: int = 3,
    resume: bool = True,
    skip_if_yaml_exists: bool = True,
    skip_if_msas_exist: bool = False,
    max_workers: Optional[int] = None,
    mani_dir: Optional[Path] = None,
) -> pd.DataFrame:

    yaml_dir.mkdir(parents=True, exist_ok=True)
    if mani_dir is not None:
        mani_dir.mkdir(parents=True, exist_ok=True)

    if max_workers is None:
        ncpu = os.cpu_count() or 8
        max_workers = max(2, min(6, ncpu // max(1, cfg.cpu_threads)))

    manifest_rows = []
    df2 = df.reset_index(drop=True)

    for i, row in df2.iterrows():
        pair_id = get_pair_id(row, i=i, pad=pad)

        pep  = normalise_seq_for_yaml(row.get("Peptide", ""))
        mhc  = normalise_seq_for_yaml(row.get("HLA_sequence", ""))
        tcra = normalise_seq_for_yaml(row.get("TCRa", ""))
        tcrb = normalise_seq_for_yaml(row.get("TCRb", ""))

        yml_path = yaml_dir / f"{pair_id}.yaml"
        try:
            yml_rel = str(yml_path.relative_to(base_dir))
        except ValueError:
            yml_rel = str(yml_path)

        manifest_rows.append({
            "pair_id": pair_id,
            "yaml_path": yml_rel,
            "pep_len": len(pep),
            "tcra_len": len(tcra),
            "tcrb_len": len(tcrb),
            "hla_len": len(mhc),
        })

    todo = []
    for i, row in df2.iterrows():
        pair_id = get_pair_id(row, i=i, pad=pad)

        yml_path = yaml_dir / f"{pair_id}.yaml"
        pair_msa_dir = msa_root / split_name / pair_id

        if resume:
            if skip_if_yaml_exists and _yaml_done(yml_path):
                continue
            if skip_if_msas_exist and _msas_done(pair_msa_dir):
                continue

        todo.append((i, row.to_dict(), split_name, str(base_dir), str(yaml_dir), str(msa_root), cfg, pad))

    print(f"[{split_name}] total rows: {len(df2)} | to process: {len(todo)} | workers: {max_workers}")

    n_ok, n_fail = 0, 0
    if todo:
        with ProcessPoolExecutor(max_workers=max_workers) as ex:
            futures = [ex.submit(_process_one_row, t) for t in todo]
            for fut in as_completed(futures):
                try:
                    pair_id, wrote, err = fut.result()
                    n_ok += int(wrote)
                except Exception as e:
                    n_fail += 1
                    print("[ERR]", repr(e))

    print(f"[{split_name}] completed: {n_ok} | failed: {n_fail} | skipped: {len(df2) - len(todo)}")

    manifest_df = pd.DataFrame(manifest_rows)
    if mani_dir is not None:
        manifest_df.to_csv(mani_dir / f"{split_name}_manifest.csv", index=False)

    return manifest_df

In [36]:
BASE_DIR = Path("/home/natasha/multimodal_model")

# Where to write new negative YAMLs
VAL_YAML_ARCH  = BASE_DIR / "data" / "val"  / "_archive_root_yamls"
TEST_YAML_ARCH = BASE_DIR / "data" / "test" / "_archive_root_yamls"

# Where to write MSAs
MSA_ROOT = BASE_DIR / "data" / "raw" / "MSA" / "jackhmmer_msas"

# -------- Paths --------
#BASE_DIR = Path("/home/natasha/multimodal_model")
DB_COMBINED = BASE_DIR / "data" / "raw" / "MSA" / "big_combo_subset_tcrs_50000_w_mhc_seqs.fasta"

# use the split DBs created
DBS = {
    "tcra": BASE_DIR / "data" / "raw" / "MSA" / "db_split" / "alpha.fasta",
    "tcrb": BASE_DIR / "data" / "raw" / "MSA" / "db_split" / "beta.fasta",
    "mhc":  BASE_DIR / "data" / "raw" / "MSA" / "db_split" / "mhc.fasta",
}
# def pick_db_for(stem: str) -> Path:
#     return DBS.get(stem, DB_COMBINED)

# OUT_ROOT = BASE_DIR / "data" / "raw" / "MSA" / "jackhmmer_msas"
# OUT_ROOT.mkdir(parents=True, exist_ok=True)

cfg = MSAConfig(
    out_root=BASE_DIR/"outputs",     # not used by functions, fine
    db_combined=DB_COMBINED,
    dbs=DBS,
    verbose=False,
    keep_intermediates=False,
    jack_iters=1,
    evalue=1e-10,
    cpu_threads=os.cpu_count() or 4,
    max_seqs=64,
    id_thr=100,
    cov_thr_tcr=50,
    cov_thr_mhc=30,
)

In [37]:
# create new YAMLs for negatives

# minimal safety: ensure required columns exist
required = ["pair_id", "Peptide", "HLA_sequence", "TCRa", "TCRb"]
for name, df in [("val", df_val_negatives), ("test", df_test_negatives)]:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing columns: {missing}")
    df["pair_id"] = df["pair_id"].astype(str)

# write YAMLs
val_neg_manifest = process_split_parallel_resume(
    df=df_val_negatives,
    split_name="val",
    base_dir=BASE_DIR,
    yaml_dir=VAL_YAML_ARCH,
    msa_root=MSA_ROOT,
    cfg=cfg,
    resume=True,
    skip_if_yaml_exists=True,  # set False if you want to overwrite
    skip_if_msas_exist=False,
    max_workers=6,
    mani_dir=None,
)

test_neg_manifest = process_split_parallel_resume(
    df=df_test_negatives,
    split_name="test",
    base_dir=BASE_DIR,
    yaml_dir=TEST_YAML_ARCH,
    msa_root=MSA_ROOT,
    cfg=cfg,
    resume=True,
    skip_if_yaml_exists=True,
    skip_if_msas_exist=False,
    max_workers=6,
    mani_dir=None,
)

[val] total rows: 2000 | to process: 2000 | workers: 6


/tmp/ipykernel_2238678/4222534594.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pair_id"] = df["pair_id"].astype(str)
/tmp/ipykernel_2238678/4222534594.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pair_id"] = df["pair_id"].astype(str)


[val] completed: 2000 | failed: 0 | skipped: 0
[test] total rows: 2000 | to process: 2000 | workers: 6
[test] completed: 2000 | failed: 0 | skipped: 0


3b) Append negatives to manifest files and then make sure symlinks are created for these YAMLs

In [38]:
# append negatives to manifest files and then make sure symlinks are created for yaml files 

import numpy as np

def append_negatives_to_manifest(split: str, df_neg: pd.DataFrame, yaml_arch: Path, manifest_path: Path):
    m = pd.read_csv(manifest_path)
    m["pair_id"] = m["pair_id"].astype(str)

    # ensure binding_flag exists
    if "binding_flag" not in m.columns:
        m["binding_flag"] = 1
    m["binding_flag"] = m["binding_flag"].fillna(1).astype(int)

    df_neg = df_neg.copy()
    df_neg["pair_id"] = df_neg["pair_id"].astype(str)

    existing = set(m["pair_id"])
    df_add = df_neg[~df_neg["pair_id"].isin(existing)].copy()

    def yaml_rel(pid: str) -> str:
        p1 = yaml_arch / f"{pid}.yaml"
        p2 = yaml_arch / f"{pid}.yml"
        if p1.exists(): return str(p1.relative_to(BASE_DIR))
        if p2.exists(): return str(p2.relative_to(BASE_DIR))
        raise FileNotFoundError(f"Missing YAML for {pid} in {yaml_arch}")

    rows = []
    for _, r in df_add.iterrows():
        pid = r["pair_id"]
        pep  = normalise_seq_for_yaml(r.get("Peptide",""))
        hla  = normalise_seq_for_yaml(r.get("HLA_sequence",""))
        tcra = normalise_seq_for_yaml(r.get("TCRa",""))
        tcrb = normalise_seq_for_yaml(r.get("TCRb",""))
        rows.append({
            "pair_id": pid,
            "yaml_path": yaml_rel(pid),
            "pep_len": len(pep),
            "tcra_len": len(tcra),
            "tcrb_len": len(tcrb),
            "hla_len": len(hla),
            "binding_flag": 0,
        })

    if not rows:
        print(f"[{split}] nothing to append")
        return

    add = pd.DataFrame(rows)
    out = pd.concat([m, add], ignore_index=True)

    if out["pair_id"].duplicated().any():
        dups = out[out["pair_id"].duplicated(keep=False)]["pair_id"].head(20).tolist()
        raise RuntimeError(f"[{split}] duplicates after append: {dups}")

    bak = manifest_path.with_suffix(manifest_path.suffix + ".bak_before_neg_append")
    Path(manifest_path).replace(bak)
    out.to_csv(manifest_path, index=False)
    print(f"[{split}] wrote {manifest_path} (backup {bak}) | new rows={len(out)}")


def ensure_chunk_symlinks(split: str, yaml_arch: Path, chunk_root: Path, per_chunk: int = 2000):
    import os
    chunk_root.mkdir(parents=True, exist_ok=True)
    yamls = sorted(list(yaml_arch.glob("*.yaml")) + list(yaml_arch.glob("*.yml")))
    if not yamls:
        print(f"[{split}] no YAMLs in {yaml_arch}")
        return

    chunks = sorted([p for p in chunk_root.iterdir() if p.is_dir() and p.name.startswith("chunk_")])
    if not chunks:
        (chunk_root/"chunk_000").mkdir(parents=True, exist_ok=True)
        chunks = [chunk_root/"chunk_000"]

    def count_yaml(d: Path) -> int:
        return len(list(d.glob("*.y*")))

    existing = set()
    for c in chunks:
        for p in list(c.glob("*.yaml")) + list(c.glob("*.yml")):
            existing.add(p.stem)

    to_link = [p for p in yamls if p.stem not in existing]
    print(f"[{split}] archive={len(yamls)} already_linked={len(existing)} to_link={len(to_link)}")

    ci = 0
    for y in to_link:
        while True:
            if ci >= len(chunks):
                new = chunk_root / f"chunk_{ci:03d}"
                new.mkdir(parents=True, exist_ok=True)
                chunks.append(new)

            if count_yaml(chunks[ci]) < per_chunk:
                break
            ci += 1

        link = chunks[ci] / y.name
        if not link.exists():
            os.symlink(y, link)

    print(f"[{split}] chunks now={len(chunks)} last={chunks[-1].name} count={count_yaml(chunks[-1])}")


# --- append negatives into official manifests
append_negatives_to_manifest(
    "val",
    df_val_negatives,
    VAL_YAML_ARCH,
    BASE_DIR/"manifests/val_manifest.csv"
)

append_negatives_to_manifest(
    "test",
    df_test_negatives,
    TEST_YAML_ARCH,
    BASE_DIR/"manifests/test_manifest.csv"
)

# --- ensure chunk symlinks exist (fills first non-full chunk)
ensure_chunk_symlinks("val",  VAL_YAML_ARCH,  BASE_DIR/"data/val/_chunks",  per_chunk=2000)
ensure_chunk_symlinks("test", TEST_YAML_ARCH, BASE_DIR/"data/test/_chunks", per_chunk=2000)

[val] wrote /home/natasha/multimodal_model/manifests/val_manifest.csv (backup /home/natasha/multimodal_model/manifests/val_manifest.csv.bak_before_neg_append) | new rows=3944
[test] wrote /home/natasha/multimodal_model/manifests/test_manifest.csv (backup /home/natasha/multimodal_model/manifests/test_manifest.csv.bak_before_neg_append) | new rows=4182
[val] archive=3944 already_linked=1944 to_link=2000
[val] chunks now=2 last=chunk_001 count=1944
[test] archive=4190 already_linked=2190 to_link=2000
[test] chunks now=3 last=chunk_002 count=190


3c) Check data integrity of all csv, manifests and YAMLs for downstream model use

In [40]:
from pathlib import Path
import pandas as pd

BASE = Path("/home/natasha/multimodal_model")

splits = {
    "train": {
        "csv": BASE/"data/train/train_df_clean.csv",
        "manifest": BASE/"manifests/train_manifest.csv",
    },
    "val": {
        "csv": BASE/"data/val/val_df_clean_pos_neg.csv",
        "manifest": BASE/"manifests/val_manifest.csv",
    },
    "test": {
        "csv": BASE/"data/test/test_df_clean_pos_neg.csv",
        "manifest": BASE/"manifests/test_manifest.csv",
    },
}

discard_rows = []

for split, paths in splits.items():
    csv_path = paths["csv"]
    man_path = paths["manifest"]

    df_csv = pd.read_csv(csv_path)
    df_man = pd.read_csv(man_path)

    df_csv["pair_id"] = df_csv["pair_id"].astype(str)
    df_man["pair_id"] = df_man["pair_id"].astype(str)

    csv_ids = set(df_csv["pair_id"])
    man_ids = set(df_man["pair_id"])

    missing_ids = sorted(csv_ids - man_ids)

    print(f"\n{split.upper()}")
    print("Rows in CSV not in manifest:", len(missing_ids))
    print("Examples:", missing_ids[:20])

    if missing_ids:
        df_missing = df_csv[df_csv["pair_id"].isin(missing_ids)].copy()
        df_missing["split"] = split
        discard_rows.append(df_missing)

        # Remove from CSV
        df_csv_clean = df_csv[~df_csv["pair_id"].isin(missing_ids)].copy()

        backup = csv_path.with_suffix(".csv.bak_before_discard")
        csv_path.rename(backup)
        df_csv_clean.to_csv(csv_path, index=False)

        print(f"Cleaned CSV written: {csv_path}")
        print(f"Backup saved: {backup}")
        print(f"New row count: {len(df_csv_clean)}")

# Save discarded rows
if discard_rows:
    df_discarded = pd.concat(discard_rows, ignore_index=True)
    discard_path = BASE/"data/discarded_seq_not_in_manifests.csv"
    df_discarded.to_csv(discard_path, index=False)
    print(f"\nDiscarded rows saved to: {discard_path}")
    print("Total discarded rows:", len(df_discarded))


TRAIN
Rows in CSV not in manifest: 12
Examples: ['pair_1058', 'pair_1190', 'pair_1229', 'pair_1331', 'pair_1338', 'pair_1351', 'pair_1432', 'pair_1436', 'pair_1542', 'pair_1753', 'pair_725', 'pair_987']
Cleaned CSV written: /home/natasha/multimodal_model/data/train/train_df_clean.csv
Backup saved: /home/natasha/multimodal_model/data/train/train_df_clean.csv.bak_before_discard
New row count: 31154

VAL
Rows in CSV not in manifest: 16
Examples: ['pair_044', 'pair_045', 'pair_046', 'pair_047', 'pair_048', 'pair_049', 'pair_050', 'pair_051', 'pair_052', 'pair_053', 'pair_1683', 'pair_1941', 'pair_1942', 'pair_1947', 'pair_666', 'pair_900']
Cleaned CSV written: /home/natasha/multimodal_model/data/val/val_df_clean_pos_neg.csv
Backup saved: /home/natasha/multimodal_model/data/val/val_df_clean_pos_neg.csv.bak_before_discard
New row count: 3944

TEST
Rows in CSV not in manifest: 8
Examples: ['pair_1058', 'pair_1190', 'pair_1331', 'pair_1338', 'pair_1351', 'pair_1412', 'pair_1753', 'pair_725']


In [41]:
from pathlib import Path
import pandas as pd
import re
import time

TS = time.strftime("%Y%m%d_%H%M%S")

def canon_pair_id(x: str, pad: int = 3) -> str:
    """
    Canonicalise pair IDs like 'pair_44' or 'pair_044' -> 'pair_044'.
    If it doesn't match, return as-is (stringified + stripped).
    """
    s = "" if x is None else str(x).strip()
    m = re.fullmatch(r"pair_(\d+)", s)
    if not m:
        return s
    return f"pair_{int(m.group(1)):0{pad}d}"

def filter_manifest_to_csv(csv_path: Path, manifest_path: Path, pad: int = 3, dry_run: bool = False):
    csv_path = Path(csv_path)
    manifest_path = Path(manifest_path)

    df_csv = pd.read_csv(csv_path)
    df_man = pd.read_csv(manifest_path)

    if "pair_id" not in df_csv.columns:
        raise ValueError(f"CSV missing pair_id column: {csv_path}")
    if "pair_id" not in df_man.columns:
        raise ValueError(f"Manifest missing pair_id column: {manifest_path}")

    # normalise IDs (does NOT change sequences; only the key used for membership)
    # df_csv = df_csv.copy()
    # df_man = df_man.copy()
    # df_csv["pair_id"] = df_csv["pair_id"].map(lambda x: canon_pair_id(x, pad=pad))
    # df_man["pair_id"] = df_man["pair_id"].map(lambda x: canon_pair_id(x, pad=pad))

    csv_ids = set(df_csv["pair_id"].astype(str))
    man_ids = set(df_man["pair_id"].astype(str))

    extra_in_manifest = sorted(man_ids - csv_ids)
    missing_in_manifest = sorted(csv_ids - man_ids)

    print("\n==============================")
    print("CSV     :", csv_path)
    print("MANIFEST:", manifest_path)
    print(f"counts: csv={len(df_csv)} (unique {len(csv_ids)}) | manifest={len(df_man)} (unique {len(man_ids)})")
    print(f"manifest not in csv: {len(extra_in_manifest)}")
    if extra_in_manifest:
        print(" examples:", extra_in_manifest[:20])
    print(f"csv not in manifest: {len(missing_in_manifest)}")
    if missing_in_manifest:
        print(" examples:", missing_in_manifest[:20])

    # Filter manifest rows to only those in CSV
    df_man_f = df_man[df_man["pair_id"].isin(csv_ids)].copy()

    # Optional: also ensure 1 row per pair_id (keep first)
    # (Only do this if you ever get duplicates; otherwise leave it)
    if df_man_f["pair_id"].duplicated().any():
        before = len(df_man_f)
        df_man_f = df_man_f.drop_duplicates(subset=["pair_id"], keep="first").copy()
        print(f"[WARN] Dropped duplicate manifest pair_id rows: {before} -> {len(df_man_f)}")

    print(f"filtered manifest rows: {len(df_man_f)}")

    if dry_run:
        print("[DRY RUN] Not writing changes.")
        return df_man_f

    backup = manifest_path.with_suffix(manifest_path.suffix + f".bak_{TS}")
    manifest_path.replace(backup)
    df_man_f.to_csv(manifest_path, index=False)
    print(f"[WROTE] {manifest_path}")
    print(f"[BACKUP] {backup}")
    return df_man_f

BASE = Path("/home/natasha/multimodal_model")

# Adjust these if your canonical CSVs differ
train_csv = BASE / "data/train/train_df_clean.csv"
val_csv   = BASE / "data/val/val_df_clean_pos_neg.csv"
test_csv  = BASE / "data/test/test_df_clean_pos_neg.csv"

train_man = BASE / "manifests/train_manifest.csv"
val_man   = BASE / "manifests/val_manifest.csv"
test_man  = BASE / "manifests/test_manifest.csv"

# Run
filter_manifest_to_csv(train_csv, train_man, pad=3, dry_run=False)
filter_manifest_to_csv(val_csv,   val_man,   pad=3, dry_run=False)
filter_manifest_to_csv(test_csv,  test_man,  pad=3, dry_run=False)


CSV     : /home/natasha/multimodal_model/data/train/train_df_clean.csv
MANIFEST: /home/natasha/multimodal_model/manifests/train_manifest.csv
counts: csv=31154 (unique 31154) | manifest=31154 (unique 31154)
manifest not in csv: 0
csv not in manifest: 0
filtered manifest rows: 31154
[WROTE] /home/natasha/multimodal_model/manifests/train_manifest.csv
[BACKUP] /home/natasha/multimodal_model/manifests/train_manifest.csv.bak_20260226_145314

CSV     : /home/natasha/multimodal_model/data/val/val_df_clean_pos_neg.csv
MANIFEST: /home/natasha/multimodal_model/manifests/val_manifest.csv
counts: csv=3944 (unique 3944) | manifest=3944 (unique 3944)
manifest not in csv: 0
csv not in manifest: 0
filtered manifest rows: 3944
[WROTE] /home/natasha/multimodal_model/manifests/val_manifest.csv
[BACKUP] /home/natasha/multimodal_model/manifests/val_manifest.csv.bak_20260226_145314

CSV     : /home/natasha/multimodal_model/data/test/test_df_clean_pos_neg.csv
MANIFEST: /home/natasha/multimodal_model/manifest

,pair_id,yaml_path,pep_len,tcra_len,tcrb_len,hla_len,binding_flag
0,pair_000,data/test/_archive_root_yamls/pair_000.yaml,10,112,117,362,1
1,pair_001,data/test/_archive_root_yamls/pair_001.yaml,11,111,111,362,1
2,pair_002,data/test/_archive_root_yamls/pair_002.yaml,11,111,111,362,1
3,pair_003,data/test/_archive_root_yamls/pair_003.yaml,11,111,111,362,1
4,pair_004,data/test/_archive_root_yamls/pair_004.yaml,11,111,111,362,1
...,...,...,...,...,...,...,...
4177,negative_pair_004185,data/test/_archive_root_yamls/negative_pair_00...,9,106,114,362,0
4178,negative_pair_004186,data/test/_archive_root_yamls/negative_pair_00...,9,111,116,362,0
4179,negative_pair_004187,data/test/_archive_root_yamls/negative_pair_00...,9,112,113,362,0
4180,negative_pair_004188,data/test/_archive_root_yamls/negative_pair_00...,9,113,113,362,0


In [ ]:
# move current val files into chunks to replicate the other structure DONE
# n.b. also need to move boltz outputs into the right format for consistency! Why didn't I do this in the first placeeee DONE


3d) Check mismatches in Data vs YAML files and check manifests

In [16]:
# compare manifest and yaml files for val

from pathlib import Path
import pandas as pd
import re

BASE = Path("/home/natasha/multimodal_model")

man_path = BASE/"manifests/val_manifest.csv"
yaml_root = BASE/"data/val/_archive_root_yamls"   # canonical store

dfm = pd.read_csv(man_path)
m_ids = set(dfm["pair_id"].astype(str))

y_files = list(yaml_root.glob("*.yaml")) + list(yaml_root.glob("*.yml"))
y_ids = set([p.stem for p in y_files])

print("manifest rows:", len(dfm), "unique:", len(m_ids))
print("yaml files:", len(y_files), "unique stems:", len(y_ids))

only_in_manifest = sorted(m_ids - y_ids)
only_in_yamls = sorted(y_ids - m_ids)

print("in manifest but missing yaml:", len(only_in_manifest))
print("in yaml but missing manifest:", len(only_in_yamls))

# show a few examples
print("examples missing yaml:", only_in_manifest[:10])
print("examples missing manifest:", only_in_yamls[:10])

manifest rows: 1938 unique: 1938
yaml files: 1944 unique stems: 1944
in manifest but missing yaml: 0
in yaml but missing manifest: 6
examples missing yaml: []
examples missing manifest: ['pair_1229', 'pair_1432', 'pair_1436', 'pair_1542', 'pair_931', 'pair_987']


In [17]:
# compare manifest vs boltz embeddings (val)

from pathlib import Path
import pandas as pd

BASE = Path("/home/natasha/multimodal_model")
out_root = BASE/"outputs/val"          # now chunked
man_path = BASE/"manifests/val_manifest.csv"

dfm = pd.read_csv(man_path)
m_ids = set(dfm["pair_id"].astype(str))

# collect all pair ids for which embeddings exist
emb_paths = list(out_root.glob("chunk_*/boltz_results_*/predictions/*/embeddings*.npz"))
emb_ids = set([p.parent.name for p in emb_paths])   # parent of embeddings file = pair_id dir

print("pairs with embeddings:", len(emb_ids))
print("manifest pairs:", len(m_ids))

missing_emb_for_manifest = sorted(m_ids - emb_ids)
extra_emb_not_in_manifest = sorted(emb_ids - m_ids)

print("manifest pairs missing embeddings:", len(missing_emb_for_manifest))
print("embeddings exist but not in manifest:", len(extra_emb_not_in_manifest))

print("examples missing embeddings:", missing_emb_for_manifest[:10])
print("examples extra embeddings:", extra_emb_not_in_manifest[:10])

pairs with embeddings: 1944
manifest pairs: 1938
manifest pairs missing embeddings: 0
embeddings exist but not in manifest: 6
examples missing embeddings: []
examples extra embeddings: ['pair_1229', 'pair_1432', 'pair_1436', 'pair_1542', 'pair_931', 'pair_987']


In [13]:
from pathlib import Path
import pandas as pd

BASE = Path("/home/natasha/multimodal_model")

# edit these once you've identified them
manifest_paths = [
    BASE/"manifests/train_manifest.csv",
    BASE/"manifests/val_manifest.csv",
    BASE/"manifests/test_manifest.csv",
]

def audit_manifest(p: Path):
    df = pd.read_csv(p)
    print("\n==", p)
    print("rows:", len(df), "unique pair_id:", df["pair_id"].nunique() if "pair_id" in df.columns else "NO pair_id")
    if "pair_id" in df.columns:
        dup = df["pair_id"][df["pair_id"].duplicated()].head(5).tolist()
        print("dups head:", dup)
    print("cols:", list(df.columns))
    return df

dfs = [audit_manifest(p) for p in manifest_paths]


== /home/natasha/multimodal_model/manifests/train_manifest.csv
rows: 31154 unique pair_id: 31154
dups head: []
cols: ['pair_id', 'yaml_path', 'pep_len', 'tcra_len', 'tcrb_len', 'hla_len']

== /home/natasha/multimodal_model/manifests/val_manifest.csv
rows: 1938 unique pair_id: 1938
dups head: []
cols: ['pair_id', 'yaml_path', 'pep_len', 'tcra_len', 'tcrb_len', 'hla_len']

== /home/natasha/multimodal_model/manifests/test_manifest.csv
rows: 2182 unique pair_id: 2182
dups head: []
cols: ['pair_id', 'yaml_path', 'pep_len', 'tcra_len', 'tcrb_len', 'hla_len']


In [22]:
from pathlib import Path
import pandas as pd
import re

BASE = Path("/home/natasha/multimodal_model")
yaml_root = BASE/"data/val/_archive_root_yamls"

df_val = pd.read_csv(BASE/"data/val/val_df_clean.csv")
df_val["pair_id"] = df_val["pair_id"].astype(str)

def norm_letters(s: str) -> str:
    if not isinstance(s, str): return ""
    return re.sub(r"[^A-Za-z]", "", s).upper()

def parse_yaml_sequences(txt: str):
    out = {}
    cur = None
    for line in txt.splitlines():
        line = line.strip()
        if line.startswith("id:"):
            cur = line.split(":",1)[1].strip()
        elif line.startswith("sequence:") and cur:
            out[cur] = line.split(":",1)[1].strip()
            cur = None
    return out

def x_wildcard_equal(a: str, b: str) -> bool:
    # compare a (csv) vs b (reconstructed), treating X in a as wildcard
    if len(a) != len(b):
        return False
    return all((ac == "X") or (ac == bc) for ac, bc in zip(a, b))

def best_match_tcrfull(csv_tcr: str, a: str, b: str) -> bool:
    csv_n = norm_letters(csv_tcr)
    a_n, b_n = norm_letters(a), norm_letters(b)

    # candidate constructions
    candidates = [
        a_n + b_n,
        b_n + a_n,
    ]
    # sometimes only one chain present in YAML
    if a_n: candidates.append(a_n)
    if b_n: candidates.append(b_n)

    for cand in candidates:
        if csv_n == cand:
            return True
        if "X" in csv_n and x_wildcard_equal(csv_n, cand):
            return True
    return False

# run check on a sample first
sample = df_val.sample(200, random_state=0)

pep_mis = []
hla_mis = []
tcr_mis = []
yaml_missing = []

for _, r in sample.iterrows():
    pid = r["pair_id"]
    yp = yaml_root/f"{pid}.yaml"
    if not yp.exists():
        yp = yaml_root/f"{pid}.yml"
    if not yp.exists():
        yaml_missing.append(pid)
        continue

    seqs = parse_yaml_sequences(yp.read_text())
    pep_csv = norm_letters(r.get("Peptide",""))
    hla_csv = norm_letters(r.get("HLA_sequence",""))
    pep_y = norm_letters(seqs.get("C",""))
    hla_y = norm_letters(seqs.get("D",""))

    if pep_csv != pep_y:
        pep_mis.append(pid)
    if hla_csv != hla_y:
        hla_mis.append(pid)

    a = seqs.get("A","")
    b = seqs.get("B","")
    if not best_match_tcrfull(r.get("TCR_full",""), a, b):
        tcr_mis.append(pid)

print("sample size:", len(sample))
print("yaml missing:", len(yaml_missing))
print("peptide mismatches:", len(pep_mis))
print("hla mismatches:", len(hla_mis))
print("TCR_full mismatches:", len(tcr_mis))
print("examples TCR_full mismatches:", tcr_mis[:10])

sample size: 200
yaml missing: 2
peptide mismatches: 0
hla mismatches: 0
TCR_full mismatches: 23
examples TCR_full mismatches: ['pair_514', 'pair_027', 'pair_1651', 'pair_417', 'pair_1145', 'pair_1358', 'pair_399', 'pair_1904', 'pair_1903', 'pair_348']


In [23]:
from pathlib import Path
import pandas as pd

BASE = Path("/home/natasha/multimodal_model")

targets = [
    ("train", BASE/"manifests/train_manifest.csv", BASE/"data/train/_archive_root_yamls"),
    ("val",   BASE/"manifests/val_manifest.csv",   BASE/"data/val/_archive_root_yamls"),
    ("test",  BASE/"manifests/test_manifest.csv",  BASE/"data/test/_archive_root_yamls"),
]

def resolve_yaml(yaml_root: Path, pid: str) -> str:
    p_yaml = yaml_root / f"{pid}.yaml"
    p_yml  = yaml_root / f"{pid}.yml"
    if p_yaml.exists():
        return str(p_yaml.relative_to(BASE))
    if p_yml.exists():
        return str(p_yml.relative_to(BASE))
    return ""

for split, man_path, yaml_root in targets:
    dfm = pd.read_csv(man_path)
    dfm["pair_id"] = dfm["pair_id"].astype(str)

    dfm["yaml_path"] = dfm["pair_id"].map(lambda pid: resolve_yaml(yaml_root, pid))

    missing = dfm[dfm["yaml_path"] == ""]
    print(f"[{split}] rows={len(dfm)} | missing_yaml={len(missing)}")

    bak = man_path.with_suffix(man_path.suffix + ".bak_yamlpath_fix")
    man_path.replace(bak)
    dfm.to_csv(man_path, index=False)
    print(f"[{split}] wrote {man_path} | backup {bak}")

    if len(missing):
        print(missing.head(20)[["pair_id","yaml_path"]])

[train] rows=31154 | missing_yaml=0
[train] wrote /home/natasha/multimodal_model/manifests/train_manifest.csv | backup /home/natasha/multimodal_model/manifests/train_manifest.csv.bak_yamlpath_fix
[val] rows=1938 | missing_yaml=0
[val] wrote /home/natasha/multimodal_model/manifests/val_manifest.csv | backup /home/natasha/multimodal_model/manifests/val_manifest.csv.bak_yamlpath_fix
[test] rows=2182 | missing_yaml=0
[test] wrote /home/natasha/multimodal_model/manifests/test_manifest.csv | backup /home/natasha/multimodal_model/manifests/test_manifest.csv.bak_yamlpath_fix


In [21]:
from pathlib import Path
import pandas as pd

BASE = Path("/home/natasha/multimodal_model")
man = BASE/"manifests/val_manifest.csv"
yaml_root = BASE/"data/val/_archive_root_yamls"

dfm = pd.read_csv(man)
dfm["pair_id"] = dfm["pair_id"].astype(str)

def resolve_yaml(pid: str) -> str:
    p_yaml = yaml_root / f"{pid}.yaml"
    p_yml  = yaml_root / f"{pid}.yml"
    if p_yaml.exists():
        return str(p_yaml.relative_to(BASE))
    if p_yml.exists():
        return str(p_yml.relative_to(BASE))
    return ""  # flag later

dfm["yaml_path"] = dfm["pair_id"].map(resolve_yaml)

missing = dfm[dfm["yaml_path"] == ""]
print("missing yaml for manifest rows:", len(missing))
print(missing.head(10)[["pair_id","yaml_path"]])

# backup + write
bak = man.with_suffix(".csv.bak_yamlpath_fix")
man.replace(bak)
dfm.to_csv(man, index=False)
print("wrote", man, "backup", bak)

missing yaml for manifest rows: 0
Empty DataFrame
Columns: [pair_id, yaml_path]
Index: []
wrote /home/natasha/multimodal_model/manifests/val_manifest.csv backup /home/natasha/multimodal_model/manifests/val_manifest.csv.bak_yamlpath_fix


In [20]:
extras = ['pair_1229','pair_1432','pair_1436','pair_1542','pair_931','pair_987']
df_val = pd.read_csv("/home/natasha/multimodal_model/data/val/val_df_clean.csv")
df_val["pair_id"] = df_val["pair_id"].astype(str)

df_val[df_val["pair_id"].isin(extras)][["pair_id","Peptide","HLA_sequence","TCR_full"]]

,pair_id,Peptide,HLA_sequence,TCR_full
350,pair_1229,TTDPSFLGRY,MAVMAPRTLLLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,AQSVTQLDSQVPVFEEAPVELRCNYSSSVSVYLFWYVQYPNQGLQL...
570,pair_1432,SPRWYFYYL,MLVMAPRTVLLLLSAALALTETWAGSHSMRYFYTSVSRPGRGEPRF...,AQKITQTQPGMFVQEKEAVTLDCTYDTSDPSYGLFWYKQPSSGEMI...
574,pair_1436,FTSDYYQLY,MAVMAPRTLLLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,AQKITQTQPGMFVQEKEAVTLDCTYDTSDPSYGLFWYKQPSSGEMI...
691,pair_1542,GILGFVFTL,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,AQTVTQSQPEMSVQEAETVTLSCTYDTSESDYYLFWYKQPPSRQMI...
1891,pair_931,LLAGIGTVPI,MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,QKEVEQDPGPLSVPEGAIVSLNCTYSNSAFQYFMWYRQYSRKGPEL...
1947,pair_987,KLGGALQAK,MAVMAPRTLLLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRF...,RKEVEQDPGPFNVPEGATVAFNCTYSNSASQSFFWYRQDCRKEPKL...


In [24]:
from pathlib import Path
import pandas as pd

BASE = Path("/home/natasha/multimodal_model")

extras = ["pair_1229","pair_1432","pair_1436","pair_1542","pair_931","pair_987"]

split = "val"
man_path = BASE/"manifests/val_manifest.csv"
df_clean_path = BASE/"data/val/val_df_clean.csv"
yaml_root = BASE/"data/val/_archive_root_yamls"
out_root = BASE/"outputs/val"

# load
m = pd.read_csv(man_path)
m["pair_id"] = m["pair_id"].astype(str)
df = pd.read_csv(df_clean_path)
df["pair_id"] = df["pair_id"].astype(str)

# helper: yaml path
def yaml_rel(pid: str) -> str:
    p1 = yaml_root/f"{pid}.yaml"
    p2 = yaml_root/f"{pid}.yml"
    if p1.exists(): return str(p1.relative_to(BASE))
    if p2.exists(): return str(p2.relative_to(BASE))
    return ""

# helper: embeddings exist?
def has_emb(pid: str) -> bool:
    return any(out_root.glob(f"chunk_*/boltz_results_*/predictions/{pid}/embeddings*.npz"))

# build rows for extras that are in df_clean and not already in manifest
need = [pid for pid in extras if pid in set(df["pair_id"]) and pid not in set(m["pair_id"])]
print("need to add:", need)

rows = []
for pid in need:
    y = yaml_rel(pid)
    if not y:
        raise RuntimeError(f"Missing YAML for {pid} under {yaml_root}")
    if not has_emb(pid):
        print(f"[WARN] no embeddings found yet for {pid} under {out_root} (will still add to manifest)")

    r = df.loc[df["pair_id"] == pid].iloc[0]
    pep = str(r.get("Peptide","") or "")
    tc  = str(r.get("TCR_full","") or "")
    hla = str(r.get("HLA_sequence","") or "")

    rows.append({
        "pair_id": pid,
        "yaml_path": y,
        "pep_len": len(pep),
        "tcra_len": 0,        # unknown from df_clean (no separate chains)
        "tcrb_len": 0,        # unknown from df_clean (no separate chains)
        "hla_len": len(hla),
        "binding_flag": 1,    # for existing positives
    })

if rows:
    add = pd.DataFrame(rows)

    # ensure binding_flag exists in manifest
    if "binding_flag" not in m.columns:
        m["binding_flag"] = 1
    m["binding_flag"] = m["binding_flag"].fillna(1).astype(int)

    # append + sort for stability
    m2 = pd.concat([m, add], ignore_index=True)
    if m2["pair_id"].duplicated().any():
        dups = m2[m2["pair_id"].duplicated(keep=False)]["pair_id"].tolist()
        raise RuntimeError(f"Duplicate pair_id after append: {dups[:20]}")

    bak = man_path.with_suffix(man_path.suffix + ".bak_before_extra_append")
    man_path.replace(bak)
    m2.to_csv(man_path, index=False)
    print("wrote:", man_path, "| backup:", bak, "| new rows:", len(m2))
else:
    print("Nothing to add.")

need to add: ['pair_1229', 'pair_1432', 'pair_1436', 'pair_1542', 'pair_931', 'pair_987']
wrote: /home/natasha/multimodal_model/manifests/val_manifest.csv | backup: /home/natasha/multimodal_model/manifests/val_manifest.csv.bak_before_extra_append | new rows: 1944
